In [1]:
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

d:\Repos\study-multi-modal-civil-complaint-classifier\.conda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "Bingsu/clip-vit-base-patch32-ko"

print("모델 로딩 중...")
model = AutoModel.from_pretrained(model_name).to(device)
processor = AutoProcessor.from_pretrained(model_name)
print("로딩 완료!")

모델 로딩 중...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


로딩 완료!


In [3]:
labels = [
    "보도나 인도가 훼손되거나 파손되어 보행에 불편함을 주는 문제",
    "인도나 횡단보도 점검 및 수리 요청",
    "도로에 구멍이나 포트홀이 있어 차량 통행에 위험한 상황",
    "아스팔트 도로 균열이나 노면 상태 불량 문제",
    "맨홀 덮개 파손이나 안전 위험 상황",
    "가로수 뿌리로 인한 보도 파손 문제",
    "도로나 보도 관련 기타 불편 사항",
    "신호등 고장이나 오작동으로 교통 흐름에 문제가 생긴 상황",
    "교통 표지판 훼손이나 불명확하여 혼란을 겪는 문제",
    "버스정류장 시설 노후화나 청결 문제",
    "자전거 도로 파손이나 통행 불편 문제",
    "교통안전 시설 점검 요청",
    "횡단보도 안전 문제",
    "교통시설 관련 기타 불편 사항",
    "쓰레기나 폐기물 무단 투기나 불법 방치 문제",
    "재활용품 분리수거나 처리 문제",
    "음식물 쓰레기 처리나 악취 문제",
    "청소 취약 지역이나 청소 필요 지역 신고",
    "길거리 폐기물이나 폐건축자재 방치 문제",
    "공공화장실 위생이나 청결 문제",
    "환경이나 쓰레기 처리 관련 기타 불편 사항",
    "공원 시설 파손으로 사용 불가능한 문제",
    "공원 조경이나 잔디 훼손 문제",
    "공원 내 쓰레기나 청소 문제",
    "가로수나 나무 훼손이나 관리 필요 상황",
    "공원 조명 고장이나 안전 문제",
    "공원 산책로나 운동시설 이용 불편 문제",
    "녹지나 공원 관련 기타 불편 사항",
    "불법 주정차로 교통 흐름이나 보행에 불편함을 주는 문제",
    "노상 적치물로 보행이나 통행에 방해가 되는 문제",
    "공공 공간 무단 점용이나 점거 문제",
    "불법 광고물 부착으로 도시 미관이나 안전 문제",
    "시민 안전이나 질서를 해치는 기타 불법 행위",
    "현수막이나 배너 무단 설치로 도시 미관이나 안전 문제",
    "벽보나 포스터 불법 부착 문제",
    "LED 전광판이나 간판 소음이나 밝기 문제",
    "광고물이 교통안전을 방해하거나 시야를 가리는 문제",
    "광고물 관련 기타 불편 사항",
    "공공기관 안내 부족이나 행정 문의",
    "주민편익 시설 부족으로 불편함을 겪는 상황",
    "소음이나 진동, 악취로 주민 생활에 불편함을 주는 문제",
    "안전사고 위험이 있어 조치가 필요한 상황",
    "민원 신청서식이나 행정절차 문의",
    "일상생활에서 겪는 기타 불편 사항"
]

In [32]:
input_text = "누군가 길거리에 쓰레기를 버리고 있어요"
image_path = "../data/raw/test_image.png"

if image_path:
    image = Image.open(image_path)
else:
    image = None

In [33]:
temperature = 0.07

In [ ]:
with torch.no_grad():
    if image is None and input_text:
        all_texts = [input_text] + labels
        text_inputs = processor.tokenizer(
            all_texts, 
            return_tensors="pt", 
            padding=True, 
            truncation=True
        ).to(device)
        
        text_outputs = model.text_model(**text_inputs)
        text_embeds = text_outputs.pooler_output
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)
        
        input_text_embed = text_embeds[0:1]
        label_embeds = text_embeds[1:]
        logits = input_text_embed @ label_embeds.T
        probs = (logits / temperature).softmax(dim=-1)
        
    elif image is not None and input_text:
        all_texts = [input_text] + labels
        inputs = processor(text=all_texts, images=image, return_tensors="pt", padding=True).to(device)
        outputs = model(**inputs)
        
        if hasattr(outputs, 'image_embeds') and hasattr(outputs, 'text_embeds'):
            image_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
            text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
            input_text_embed = text_embeds[0:1]
            label_embeds = text_embeds[1:]
            combined_embed = (image_embeds + input_text_embed) / 2
            combined_embed = combined_embed / combined_embed.norm(dim=-1, keepdim=True)
            logits = combined_embed @ label_embeds.T
        elif hasattr(outputs, 'logits_per_image'):
            logits = outputs.logits_per_image
        else:
            raise ValueError(f"모델 출력 구조를 확인하세요. 출력 타입: {type(outputs)}")
        
        probs = (logits / temperature).softmax(dim=-1)
        
    elif image is not None:
        inputs = processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)
        outputs = model(**inputs)
        
        if hasattr(outputs, 'image_embeds') and hasattr(outputs, 'text_embeds'):
            image_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
            text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
            logits = image_embeds @ text_embeds.T
        elif hasattr(outputs, 'logits_per_image'):
            logits = outputs.logits_per_image
        else:
            raise ValueError(f"모델 출력 구조를 확인하세요. 출력 타입: {type(outputs)}")
        
        probs = (logits / temperature).softmax(dim=-1)
    else:
        raise ValueError("이미지와 텍스트 중 적어도 하나는 입력해야 합니다.")

In [35]:
top_k = 5
top_probs, top_indices = torch.topk(probs[0], top_k)

print("=" * 60)
print("상위 예측 결과:")
print("=" * 60)
for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
    print(f"{i+1}. {labels[idx.item()]}: {prob.item():.4f}")

pred_idx = probs.argmax().item()
print("=" * 60)
print(f"예측 민원 유형: {labels[pred_idx]}")
print(f"확률: {probs[0][pred_idx].item():.4f}")
print("=" * 60)

상위 예측 결과:
1. 쓰레기나 폐기물 무단 투기나 불법 방치 문제: 0.0594
2. 길거리 폐기물이나 폐건축자재 방치 문제: 0.0560
3. 공원 내 쓰레기나 청소 문제: 0.0508
4. 환경이나 쓰레기 처리 관련 기타 불편 사항: 0.0456
5. 청소 취약 지역이나 청소 필요 지역 신고: 0.0330
예측 민원 유형: 쓰레기나 폐기물 무단 투기나 불법 방치 문제
확률: 0.0594
